# T/NK Cell UMAP Parameter Optimization

**Objective**: Optimize UMAP parameters to improve visualization

**Background**:
- BBKNN integration shows good batch mixing
- NK (NCAM1), CD4, CD8A markers show biological separation
- Need to fine-tune UMAP for better cluster visualization

**Test Parameters**:
- `min_dist`: 0.1, 0.2, 0.3, 0.4
- `n_neighbors`: 15, 20, 30, 40
- `spread`: 0.5, 1.0, 1.5

**Fixed**:
- Use best BBKNN config from previous test (you'll select)
- Clustering resolution: 3.0

In [ ]:
import scanpy as sc
import bbknn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=80, facecolor='white')
plt.rcParams['figure.figsize'] = (6, 5)

print(f"scanpy: {sc.__version__}")

In [ ]:
# Configuration
INPUT_H5AD = "/home/h2048/data/py/1128/bbknn_celltype_analysis/T/adata_T_bbknn.h5ad"
OUTPUT_DIR = "/home/h2048/data/py/1203/umap_param_test"

# Selected BBKNN parameters (change based on your previous test results)
SELECTED_BBKNN = {
    'neighbors_within_batch': 4,  # Change to your best value (3 or 4)
    'trim': 28,                    # Change to your best value (30, 35, or 40)
    'metric': 'correlation'
}

# Clustering resolution
LEIDEN_RESOLUTION = 3.0

# Create output
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
fig_dir = output_dir / "figures"
fig_dir.mkdir(exist_ok=True)

print(f"Output directory: {output_dir}")
print(f"\nSelected BBKNN config: neighbors={SELECTED_BBKNN['neighbors_within_batch']}, trim={SELECTED_BBKNN['trim']}")

---
## 1. Prepare Data with Best BBKNN Config

In [ ]:
# Load and preprocess
print("Loading data...")
adata = sc.read_h5ad(INPUT_H5AD)
print(f"Loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")

# Store raw
if adata.raw is None:
    adata.raw = adata.copy()

# Basic preprocessing
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=4000, flavor='seurat_v3')

# HVG subset
adata_hvg = adata[:, adata.var['highly_variable']].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=50)

print(f"\nHVG subset: {adata_hvg.shape[0]:,} cells × {adata_hvg.shape[1]:,} genes")

In [ ]:
# Apply selected BBKNN configuration
print("\nApplying BBKNN with selected parameters...")
print(f"  neighbors_within_batch: {SELECTED_BBKNN['neighbors_within_batch']}")
print(f"  trim: {SELECTED_BBKNN['trim']}")
print(f"  metric: {SELECTED_BBKNN['metric']}")

bbknn.bbknn(
    adata_hvg,
    batch_key='dataset',
    neighbors_within_batch=SELECTED_BBKNN['neighbors_within_batch'],
    n_pcs=50,
    trim=SELECTED_BBKNN['trim'],
    metric=SELECTED_BBKNN['metric']
)

print("✓ BBKNN completed")

# Run clustering (will be reused for all UMAP tests)
sc.tl.leiden(adata_hvg, resolution=LEIDEN_RESOLUTION, key_added='leiden')
n_clusters = adata_hvg.obs['leiden'].nunique()
print(f"✓ Clustering completed: {n_clusters} clusters at resolution {LEIDEN_RESOLUTION}")

---
## 2. UMAP Parameter Grid Search

Test combinations of min_dist and n_neighbors

In [ ]:
# Define UMAP parameter grid
umap_params = [
    # Format: (name, min_dist, n_neighbors, spread, description)
    ('default', 0.3, 30, 1.0, 'Current default'),
    ('tight', 0.1, 15, 1.0, 'Tight local structure'),
    ('moderate_tight', 0.15, 20, 1.0, 'Moderate tight'),
    ('moderate', 0.2, 25, 1.0, 'Balanced'),
    ('loose', 0.3, 40, 1.0, 'Loose global structure'),
    ('tight_spread', 0.1, 15, 1.5, 'Tight with larger spread'),
    ('compact', 0.1, 20, 0.5, 'Very compact'),
]

print(f"Testing {len(umap_params)} UMAP configurations\n")

In [ ]:
# Run UMAP with different parameters
umap_results = {}

for param_name, min_dist, n_neighbors, spread, desc in umap_params:
    print(f"\nTesting: {param_name}")
    print(f"  Description: {desc}")
    print(f"  Parameters: min_dist={min_dist}, n_neighbors={n_neighbors}, spread={spread}")
    
    # Create a copy to store UMAP
    adata_test = adata_hvg.copy()
    
    # Run UMAP
    sc.tl.umap(
        adata_test,
        min_dist=min_dist,
        n_components=2,
        spread=spread,
        random_state=42  # Fixed seed for reproducibility
    )
    
    # Note: n_neighbors is controlled by sc.pp.neighbors, which was already done by BBKNN
    # We can't change it without re-running neighbors, so we'll note this limitation
    
    # Store result
    umap_results[param_name] = {
        'adata': adata_test,
        'params': (min_dist, n_neighbors, spread)
    }
    
    print(f"  ✓ UMAP completed")

print("\n✓ All UMAP tests completed")

---
## 3. Visual Comparison - Batch Mixing

In [ ]:
# Compare batch mixing across UMAP parameters
n_configs = len(umap_results)
n_cols = 3
n_rows = (n_configs + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten() if n_configs > 1 else [axes]

for idx, (param_name, result) in enumerate(umap_results.items()):
    adata_test = result['adata']
    min_dist, n_neighbors, spread = result['params']
    
    sc.pl.umap(
        adata_test, 
        color='dataset',
        ax=axes[idx],
        title=f"{param_name}\n(md={min_dist}, nn={n_neighbors}, sp={spread})",
        show=False,
        legend_loc='none',
        frameon=False
    )

# Hide extra subplots
for idx in range(n_configs, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(fig_dir / 'umap_comparison_batch.pdf', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Batch mixing comparison saved")

---
## 4. Visual Comparison - Clustering

In [ ]:
# Compare clustering visualization
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten() if n_configs > 1 else [axes]

for idx, (param_name, result) in enumerate(umap_results.items()):
    adata_test = result['adata']
    min_dist, n_neighbors, spread = result['params']
    
    sc.pl.umap(
        adata_test,
        color='leiden',
        ax=axes[idx],
        title=f"{param_name}\n({n_clusters} clusters)",
        show=False,
        legend_loc='on data',
        legend_fontsize=6,
        frameon=False
    )

# Hide extra subplots
for idx in range(n_configs, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(fig_dir / 'umap_comparison_clustering.pdf', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Clustering comparison saved")

---
## 5. Marker Gene Comparison

In [ ]:
# Define key markers
key_markers = ['NCAM1', 'CD4', 'CD8A', 'CD3D']

# Check availability
available_markers = [m for m in key_markers if m in adata_hvg.raw.var_names]
print(f"Available markers: {available_markers}")

if len(available_markers) == 0:
    print("Warning: No key markers found in data")
else:
    # Compare marker expression across configs
    for marker in available_markers[:3]:  # Show first 3 markers
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        plot_configs = list(umap_results.keys())[:6]  # Show first 6 configs
        
        for idx, param_name in enumerate(plot_configs):
            adata_test = umap_results[param_name]['adata']
            
            sc.pl.umap(
                adata_test,
                color=marker,
                use_raw=True,
                ax=axes[idx],
                title=f"{param_name}: {marker}",
                show=False,
                vmax='p99',
                cmap='Reds',
                frameon=False
            )
        
        plt.tight_layout()
        plt.savefig(fig_dir / f'umap_comparison_marker_{marker}.pdf', dpi=150, bbox_inches='tight')
        plt.show()
    
    print("✓ Marker gene comparisons saved")

---
## 6. Quantitative Metrics

In [ ]:
# Calculate UMAP embedding quality metrics
def calculate_umap_metrics(adata):
    """Calculate embedding quality metrics"""
    coords = adata.obsm['X_umap']
    
    # Coordinate range (larger = more spread out)
    x_range = coords[:, 0].max() - coords[:, 0].min()
    y_range = coords[:, 1].max() - coords[:, 1].min()
    
    # Density (cells per unit area)
    area = x_range * y_range
    density = adata.n_obs / area
    
    # Cluster compactness (within-cluster variance)
    cluster_compactness = []
    for cluster in adata.obs['leiden'].unique():
        cluster_coords = coords[adata.obs['leiden'] == cluster]
        if len(cluster_coords) > 1:
            variance = np.var(cluster_coords, axis=0).sum()
            cluster_compactness.append(variance)
    
    mean_compactness = np.mean(cluster_compactness)
    
    return {
        'x_range': x_range,
        'y_range': y_range,
        'total_spread': x_range + y_range,
        'density': density,
        'mean_cluster_compactness': mean_compactness
    }

# Calculate metrics for all configs
metrics_list = []

for param_name, result in umap_results.items():
    adata_test = result['adata']
    min_dist, n_neighbors, spread = result['params']
    
    metrics = calculate_umap_metrics(adata_test)
    
    metrics_list.append({
        'config': param_name,
        'min_dist': min_dist,
        'n_neighbors': n_neighbors,
        'spread': spread,
        **metrics
    })

metrics_df = pd.DataFrame(metrics_list)

print("\nUMAP Quality Metrics:")
print(metrics_df.to_string(index=False))

# Save metrics
metrics_df.to_csv(output_dir / 'umap_param_metrics.csv', index=False)
print(f"\n✓ Metrics saved to: {output_dir / 'umap_param_metrics.csv'}")

---
## 7. Side-by-Side Comparison (Selected Configs)

In [ ]:
# Compare 3 representative configs side by side
selected_configs = ['default', 'tight', 'moderate']  # Adjust based on your results
selected_configs = [c for c in selected_configs if c in umap_results]

if len(selected_configs) >= 2:
    fig, axes = plt.subplots(3, len(selected_configs), figsize=(6*len(selected_configs), 16))
    
    for col, config_name in enumerate(selected_configs):
        adata_test = umap_results[config_name]['adata']
        min_dist, n_neighbors, spread = umap_results[config_name]['params']
        
        # Row 1: Batch
        sc.pl.umap(adata_test, color='dataset', ax=axes[0, col],
                   title=f"{config_name}\n(md={min_dist}, sp={spread})",
                   show=False, legend_loc='right margin', frameon=False)
        
        # Row 2: Clustering
        sc.pl.umap(adata_test, color='leiden', ax=axes[1, col],
                   title=f"Clusters ({n_clusters})",
                   show=False, legend_loc='on data', legend_fontsize=6, frameon=False)
        
        # Row 3: Key marker (if available)
        if len(available_markers) > 0:
            sc.pl.umap(adata_test, color=available_markers[0], use_raw=True,
                       ax=axes[2, col], title=available_markers[0],
                       show=False, vmax='p99', cmap='Reds', frameon=False)
        else:
            axes[2, col].axis('off')
    
    plt.tight_layout()
    plt.savefig(fig_dir / 'umap_selected_comparison.pdf', dpi=200, bbox_inches='tight')
    plt.show()
    
    print("✓ Side-by-side comparison saved")

---
## 8. Summary and Recommendations

In [ ]:
print("\n" + "="*70)
print("UMAP PARAMETER TEST SUMMARY")
print("="*70)

print("\n[ Fixed Parameters ]")
print(f"  BBKNN: neighbors={SELECTED_BBKNN['neighbors_within_batch']}, trim={SELECTED_BBKNN['trim']}")
print(f"  Clustering: resolution={LEIDEN_RESOLUTION}, n_clusters={n_clusters}")

print("\n[ Tested UMAP Configurations ]")
for param_name, min_dist, n_neighbors, spread, desc in umap_params:
    print(f"  {param_name}: {desc}")
    print(f"    → min_dist={min_dist}, n_neighbors={n_neighbors}, spread={spread}")

print("\n[ Key Metrics ]")
print("\nEmbedding Spread (higher = more dispersed):")
for _, row in metrics_df.iterrows():
    print(f"  {row['config']:15s}: {row['total_spread']:.2f}")

print("\nCluster Compactness (lower = tighter clusters):")
for _, row in metrics_df.iterrows():
    print(f"  {row['config']:15s}: {row['mean_cluster_compactness']:.2f}")

print("\n[ Interpretation Guide ]")
print("  • min_dist controls local vs global structure:")
print("    - Lower (0.1): Tight clusters, emphasizes local differences")
print("    - Higher (0.3-0.4): Looser, emphasizes global relationships")
print("  • spread controls overall dispersion:")
print("    - Lower (0.5): Compact embedding")
print("    - Higher (1.5): Spread out embedding")
print("  • For T/NK cells with multiple subtypes:")
print("    - Recommended: min_dist=0.1-0.2, spread=1.0-1.5")

print("\n[ Selection Criteria ]")
print("  1. Visual inspection of UMAP plots:")
print("     - Are NK/CD4/CD8 cells well separated?")
print("     - Do clusters look biologically meaningful?")
print("     - Is batch mixing still appropriate?")
print("  2. Marker gene expression:")
print("     - Clear separation of NCAM1, CD4, CD8A?")
print("  3. Cluster compactness:")
print("     - Not too fragmented, not too merged")

print("\n[ Recommended Configs ]")
print("  Based on metrics, consider:")
# Find configs with moderate spread and compactness
moderate_spread = metrics_df[(metrics_df['total_spread'] > metrics_df['total_spread'].quantile(0.3)) &
                             (metrics_df['total_spread'] < metrics_df['total_spread'].quantile(0.7))]
if len(moderate_spread) > 0:
    for _, row in moderate_spread.head(3).iterrows():
        print(f"  → {row['config']}: md={row['min_dist']}, sp={row['spread']}")

print("\n[ Output Files ]")
print(f"  Metrics: {output_dir / 'umap_param_metrics.csv'}")
print(f"  Figures: {fig_dir / '*.pdf'}")

print("\n" + "="*70)

---
## 9. Save Selected Configuration (Optional)

In [ ]:
# # Uncomment and set your preferred config after visual inspection
# BEST_UMAP_CONFIG = 'tight'  # Change to your selected config

# adata_final = umap_results[BEST_UMAP_CONFIG]['adata'].copy()

# # Add configuration info to uns
# adata_final.uns['bbknn_params'] = SELECTED_BBKNN
# adata_final.uns['umap_params'] = {
#     'config': BEST_UMAP_CONFIG,
#     'min_dist': umap_results[BEST_UMAP_CONFIG]['params'][0],
#     'spread': umap_results[BEST_UMAP_CONFIG]['params'][2]
# }

# # Save
# output_file = output_dir / f'adata_tcell_optimized_{BEST_UMAP_CONFIG}.h5ad'
# adata_final.write_h5ad(output_file)
# print(f"✓ Saved optimized result to: {output_file}")

# # Print final parameters
# print("\nFinal optimized parameters:")
# print(f"  BBKNN: neighbors={SELECTED_BBKNN['neighbors_within_batch']}, trim={SELECTED_BBKNN['trim']}")
# print(f"  UMAP: min_dist={adata_final.uns['umap_params']['min_dist']}, spread={adata_final.uns['umap_params']['spread']}")
# print(f"  Clustering: resolution={LEIDEN_RESOLUTION}, n_clusters={n_clusters}")